In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Build Drivers Dimension
# MAGIC
# MAGIC 1. Read silver `drivers` table
# MAGIC 1. Read gold `ref_nationality_region` table
# MAGIC 1. Join the data from `drivers` with `ref_nationality_region` using `nationality`
# MAGIC 1. Select the required columns
# MAGIC     - drivers.driver_id
# MAGIC     - drivers.driver_name
# MAGIC     - drivers.date_of_birth
# MAGIC     - drivers.nationality
# MAGIC     - ref_nationality_region.region
# MAGIC 1. Write the transformed data to gold `dim_drivers` table
# MAGIC

In [0]:
%run ../00-common/01.environment_config

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_drivers"

In [0]:
drivers_df = spark.table(f"{catalog_name}.{silver_schema}.drivers")
nationality_region_df = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")



In [0]:
joined_df = drivers_df \
    .join(
        nationality_region_df,
        drivers_df.nationality == nationality_region_df.nationality,
        "left"
    ) \
    .select(
        drivers_df.driver_id,
        drivers_df.driver_name,
        drivers_df.date_of_birth,
        nationality_region_df.region.alias("nationality_region")
    )

In [0]:
( joined_df
.write
.format("delta")
.mode("overwrite")
.saveAsTable(target_table) )

In [0]:
spark.table(target_table).display()